In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord

# ============================================================
# Rutas
# ============================================================

home = Path.home()

roman_rubin_dir = (
    home
    / "microlensing"
    / "simulation_Rubin"
    / "roman_rubin"
)

data_file = (
    home
    / "ulensing_degenerate_models"
    / "Parallax_LSST"
    / "data_sedighe"
    / "LSSTMONTS.dat"
)

columns_file = (
    home
    / "ulensing_degenerate_models"
    / "Parallax_LSST"
    / "data_sedighe"
    / "columns"
)

output_dir = (
    home
    / "ulensing_degenerate_models"
    / "Parallax_LSST"
    / "runs"
    / "Sedighe_first_event"
)

ephemerides = roman_rubin_dir / "ephemerides" / "Roman_positions.npy"

sys.path.insert(0, str(roman_rubin_dir))

from functions_roman_rubin import sim_fit

In [2]:
columns = [
    line.strip()
    for line in columns_file.read_text().splitlines()
    if line.strip()
]

# El archivo nuevo tiene una columna alpha adicional.
n_data_columns = len(data_file.open().readline().split())

if n_data_columns == len(columns) + 1:
    columns.append("alpha")

row = pd.read_csv(
    data_file,
    sep=r"\s+",
    names=columns,
    header=None,
    nrows=1,
).iloc[0]

row

Number_lensing                     1.000000
NL1                                0.000000
NL2                                0.000000
Galactic latitude (degree)       -20.000000
Galactic Longitude (degree)      260.000000
                                   ...     
Number of data points            547.000000
Delta chi2                     19727.014000
Rho_stars                      43991.917364
N_stars                        68772.020763
alpha                             80.000000
Name: 0, Length: 69, dtype: float64

In [3]:
l_deg = float(row["Galactic Longitude (degree)"])
b_deg = float(row["Galactic latitude (degree)"])

piE = float(row["Normalized Parallax amplitude"])
xi = np.deg2rad(float(row["alpha"]))

galactic_vector = SkyCoord(
    l=l_deg * u.deg,
    b=b_deg * u.deg,
    pm_l_cosb=piE * np.cos(xi) * u.mas / u.yr,
    pm_b=piE * np.sin(xi) * u.mas / u.yr,
    frame="galactic",
)

icrs_vector = galactic_vector.icrs

piEE = float(icrs_vector.pm_ra_cosdec.value)
piEN = float(icrs_vector.pm_dec.value)

t0_zero_jd = 2460413.013828608

custom_system = {
    "t0": (
        t0_zero_jd
        + float(row["time of closest approach (days)"])
    ),
    "u0": float(row["Lens impact parameter"]),
    "tE": float(row["Einstein crossing time (days)"]),
    "rho": float(row["Normalized source radius"]),
    "piEN": piEN,
    "piEE": piEE,

    "u": float(row["Apparent magnitude of source star in u-band"]),
    "g": float(row["Apparent magnitude of source star in g-band"]),
    "r": float(row["Apparent magnitude of source star in r-band"]),
    "i": float(row["Apparent magnitude of source star in i-band"]),
    "z": float(row["Apparent magnitude of source star in z-band"]),
    "Y": float(row["Apparent magnitude of source star in y-band"]),

    "ra": float(row["Right ascention  (degree)"]),
    "dec": float(row["Declination (degree)"]),

    # Metadatos opcionales
    "mass": float(row["Lens_mass (solar mass)"]),
    "D_L_kpc": float(row["Lens_Distance (kpc)"]),
    "D_S_kpc": float(row["Source_distance (kpc)"]),
    "thetaE_mas": float(row["Angular Einstein radius (mas)"]),
    "mu_rel_catalog_masyr": float(
        row[
            "lens-source relative angular velocity (mas/years)"
        ]
    ),

    # Quedan disponibles como metadatos.
    "blend_u": float(row["Blending factor in u-band"]),
    "blend_g": float(row["Blending factor in g-band"]),
    "blend_r": float(row["Blending factor in r-band"]),
    "blend_i": float(row["Blending factor in i-band"]),
    "blend_z": float(row["Blending factor in z-band"]),
    "blend_y": float(row["Blending factor in y-band"]),
}

custom_system

{'t0': 2463119.276238608,
 'u0': 0.302151,
 'tE': 28.704443,
 'rho': 0.0013931,
 'piEN': 0.3062473223245454,
 'piEE': 1.9928080411454276,
 'u': 16.72189,
 'g': 32.6337,
 'r': 17.45916,
 'i': 23.23194,
 'z': 27.12949,
 'Y': 19.41906,
 'ra': 103.69,
 'dec': -50.2,
 'mass': 0.12384,
 'D_L_kpc': 0.216,
 'D_S_kpc': 1.89,
 'thetaE_mas': 2.0337887,
 'mu_rel_catalog_masyr': 0.06242,
 'blend_u': 1.0,
 'blend_g': 1.0,
 'blend_r': 1.0,
 'blend_i': 1.0,
 'blend_z': 1.0,
 'blend_y': 1.0}

In [4]:
model_dir = output_dir / "models"
fit_dir = output_dir / "fits"
results_dir = output_dir / "results"

for directory in [model_dir, fit_dir, results_dir]:
    directory.mkdir(parents=True, exist_ok=True)

result = sim_fit(
    i=0,
    system_type="CUSTOM",
    model="FSPL",
    algo="TRF",

    path_TRILEGAL_set=None,
    path_GENULENS_set=None,

    path_to_save_model=str(model_dir) + "/",
    path_to_save_fit=str(fit_dir) + "/",
    path_ephemerides=str(ephemerides),
    path_to_save_results=str(results_dir) + "/",

    catalog_mode="custom_system",
    custom_system=custom_system,

    pair_catalog=None,
    path_pair_catalog=None,
    param_samplers=None,

    use_roman=False,
    use_rubin=True,
    rubin_pointing_mode="source",

    fit_model="FSPL",
    fit_parallax=False,

    apply_detection_criteria=False,
    return_data=True,
)

result

[sim_fit] catalog_mode='custom_system'
[sim_fit] No se usaron catálogos ni emparejamiento.
[sim_fit] Parámetros verdaderos: {'t0': 2463119.276238608, 'u0': 0.302151, 'tE': 28.704443, 'rho': 0.0013931, 'piEN': 0.3062473223245454, 'piEE': 1.9928080411454276}
[sim_event] Rubin-only source coordinates passed to MAF
[sim_event] source RA  = 103.6900000000 deg
[sim_event] source Dec = -50.2000000000 deg
[MAF] Source-coordinate Rubin-only field
[MAF] source RA  = 103.6900000000 deg
[MAF] source Dec = -50.2000000000 deg
[MAF] MAF RA     = 103.6900000000 deg
[MAF] MAF Dec    = -50.2000000000 deg
[MAF] cache mode = exact
[MAF] cache cell = None


Parallax(Full) estimated for the telescope u: SUCCESS
Parallax(Full) estimated for the telescope g: SUCCESS
Parallax(Full) estimated for the telescope r: SUCCESS
Parallax(Full) estimated for the telescope i: SUCCESS
Parallax(Full) estimated for the telescope z: SUCCESS
Parallax(Full) estimated for the telescope y: SUCCESS
Parallax(Full) estimated for the telescope u: SUCCESS
Parallax(Full) estimated for the telescope g: SUCCESS
Parallax(Full) estimated for the telescope r: SUCCESS
Parallax(Full) estimated for the telescope i: SUCCESS
Parallax(Full) estimated for the telescope z: SUCCESS
Parallax(Full) estimated for the telescope y: SUCCESS
Detection criterion disabled. The event has valid data and will be fitted.
Criteria satisfied. Save the simulated light-curve.
chi2_true: 324.935029041323
n_data_true: 355
Saving complete Simulation...
File saved: /home/anibal/ulensing_degenerate_models/Parallax_LSST/runs/Sedighe_first_event/models/Event_0.h5
Start the fit using active telescopes:
ch

KeyError: 'mu_rel'

In [ ]:
import functions_roman_rubin as frr

print(frr.__file__)